In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
inventoryData2 = [
('Laptop', 20, {'color': 'silver', 'weight': 2.5}),
('Phone', 50, {'color': 'black', 'weight': 0.5}),
('Tablet', 30, {'color': 'white', 'weight': 0.8}),
('Chair', 10, {'color': 'brown', 'weight': 3.0}),
('Printer', 15, {'color': 'gray', 'weight': 7.5}),
]
propSchema = StructType[
    (
        StructField('color', StringType(),True),
        StructField('weight', DoubleType(),True)
    )
]

schema2 = StructType(
    [
        StructField('name', StringType(),True),
        StructField('quantity', IntegerType(),True),
        StructField('properties', propSchema, True)
       
    ]
)
df2 = spark.createDataFrame(inventoryData2, schema2)

In [0]:
#Question 1 : Get silver and weight in separate columns

inventoryData = [
('Laptop', 20, {'color': 'silver', 'weight': 2.5}),
('Phone', 50, {'color': 'black', 'weight': 0.5}),
('Tablet', 30, {'color': 'white', 'weight': 0.8}),
('Chair', 10, {'color': 'brown', 'weight': 3.0}),
('Printer', 15, {'color': 'gray', 'weight': 7.5}),
]
schema = ["Product", "Quantity", "Properties"]
df=spark.createDataFrame(inventoryData, schema)
df.display()

In [0]:
df_1 = df.withColumn("color",df.Properties["color"]).withColumn("weight",df.Properties["weight"]).drop("Properties")
display(df_1)

In [0]:
# Question 2: 
data = [
    ("tokya", "M"),
    ("tokya", "F"),
    ("tokya", "M"),
    ("tokya", "M"),
    ("seoul", "M"),
    ("seoul", "F"),
    ("seoul", "F"),
    ("seoul", "M"),
    ("jakarta", "M"),
    ("jakarta", "M")
]

columns = ["city", "gender"]

df = spark.createDataFrame(data, columns)  
display(df)

Expected Output

| city | total_count | Male_count | Female_count |
|---|---|---|---|
| tokya | 4 | 3 | 1 |
| seoul | 4 | 2 | 2 |
| jakarta | 2 | 2 | 0 |

In [0]:
df.createOrReplaceTempView("temp")

In [0]:
%sql
select city, 
sum(case when gender = 'M' then 1 else 0 end) as Male_count,
sum(case when gender = 'F' then 1 else 0 end) as Female_count,
sum(case when gender = 'M' or gender = 'F' then 1 else 0 end) as total_count
from temp
group by city

In [0]:
(df.groupBy("city").agg(
                        count("gender").alias("Total_Count"),
                        count(when(col("gender")=="M",1)).alias("Male_Count"),
                        count(when(col("gender")=='F',1)).alias("Female_Count")
)).display()

In [0]:
# 2. Define the exact schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("email", StringType(), True)
])

# 3. Create dummy data covering all Good and Rejected scenarios
data = [
    # 🟢 GOOD RECORD: Passes all conditions
    (1, "Alice", 25, "Active", "alice@example.com"),

    # 🟢 GOOD RECORD: Passes all conditions
    (7, "Aliceiop", 25, "Active", "aliceiop@example.com"),      
    
    # 🔴 REJECTED: Fails 'valid_age' (16 < 18)
    (2, "Bob", 16, "Pending", "bob@example.com"),         
    
    # 🔴 REJECTED: Fails 'valid_status' ('Closed' is not Active/Pending)
    (3, "Charlie", 30, "Closed", "charlie@example.com"),  
    
    # 🔴 REJECTED: Fails 'email_not_null' (email is None/NULL)
    (4, "Diana", 45, "Active", None),                     
    
    # 🔴 REJECTED: Fails ALL conditions at once
    (5, "Eve", 15, "Suspended", None),                    
    
    # 🔴 REJECTED: Null evaluation (age is NULL, so age >= 18 evaluates to NULL)
    (6, "Frank", None, "Active", "frank@example.com")     
]

# 4. Create and display the DataFrame
df = spark.createDataFrame(data, schema)
display(df)

In [0]:
dq_rules = {
    "valid_age" : "age >= 18",
    "valid_status" : "status IN ('Active','Pending')",
    "email_not_null" : "email is not NULL",
    "age_not_null" : "age is not NULL"
}

sql_expr = " AND ".join(f"({rule})" for rule in dq_rules.values())

df_evaluated = df.withColumn("is_good_record",expr(sql_expr))
print(sql_expr)
clean_df = df_evaluated.filter(col("is_good_record") == True)
display(clean_df)
quarantine_df = df_evaluated.filter(col("is_good_record") == False)
display(quarantine_df)

In [0]:
df.where(expr("(age >= 18) AND (status IN ('Active','Pending')) AND (email is not NULL) AND (age is not NULL)")).display()

df.withColumn("is_good_record",(expr("(age >= 18) AND (status IN ('Active','Pending')) AND (email is not NULL) AND (age is not NULL)"))).display()


In [0]:
#age >= 18 AND status IN ('Active','Pending') AND email is not NULL AND age is not NULL
#(age >= 18) AND (status IN ('Active','Pending')) AND (email is not NULL) AND (age is not NULL)

In [0]:
(df.withColumn("is_bad_record",when((col("age")<18) | (col("age").isNull()) | (col("email").isNull()) | ~(col("status").isin('Active','Pending')),False).otherwise(True)).display())

In [0]:
failure_reasons = []

# Iterate through the dictionary to check each rule independently
for rule_name, rule_expr in dq_rules.items():
    # If the rule fails or is null, return the rule's name. Otherwise, return NULL.
    reason_col = when(~expr(rule_expr) | expr(rule_expr).isNull(), lit(rule_name)).otherwise(lit(None))
    failure_reasons.append(reason_col)
print(reason_col)

In [0]:
# Combine results into an array and remove the NULLs
df_with_reasons = df.withColumn("failed_rules", array_remove(array(*failure_reasons), lit(None)))

# Good records have an empty array of failed rules
good_df = df_with_reasons.filter(size("failed_rules") == 0).drop("failed_rules")

# Rejected records have 1 or more failed rules (the array tells you exactly which ones failed)
rejected_df = df_with_reasons.filter(size("failed_rules") > 0)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
schema = StructType([
    StructField("revenue", IntegerType(), True),
    StructField("date", StringType(), True)
])

# Define the data
data = [
    (3000, "22-may"),
    (5000, "23-may"),
    (5000, "25-may"),
    (10000, "22-june"),
    (1250, "03-july")
]
input_df = spark.createDataFrame(data, schema)
display(input_df)

| revenue | month | cummulativeSum |
|---------|-------|----------------|
| 1250    | July  | 1250           |
| 10000   | June  | 10000          |
| 3000    | May   | 3000           |
| 5000    | May   | 13000          |
| 5000    | May   | 13000          |

In [0]:
schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True)
])

data = [
    (1, "John", "IT", 60000),
    (2, "Emma", "HR", 55000),
    (3, "Alex", "IT", 80000),
    (4, "David", "Sales", 45000),
    (5, "Sarah", "HR", 75000),
    (6, "Michael", "Sales", 50000)
]
df = spark.createDataFrame(data, schema)
display(df)

In [0]:
global_avg_salary = df.select(avg("salary")).collect()[0][0]

In [0]:
df.filter(col("salary")>global_avg_salary).display()

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("department")

In [0]:
df.withColumn("avg_salary_dept_wise",avg(col("salary").over(window_spec))).